In [ ]:
import numpy as np
import pandas as pd
import geopandas as gp
import requests

Nous commençons par télécharger les données à partir de data.gouv.

In [27]:
url_deces = f"https://www.data.gouv.fr/api/1/datasets/r/0b0c0fbc-af9a-4b9d-9949-b171e7e6fe4a"
deces = pd.read_csv(url_deces, sep=';', dtype=str)
deces.head()

,code_lieu_deces,deces_2008,deces_2009,deces_2010,deces_2011,deces_2012,deces_2013,deces_2014,deces_2015,deces_2016,...,deces_2024,deces_2025,total_2008_2025,moy_annuelle,annee_pic,deces_annee_pic,evol_pct_2008_vs_2025,tendance,surplus_covid_2020,surplus_pct_covid_2020
0,31555,4559,4647,4650,4763,4924,4877,4981,5208,5153,...,5433,5626,92323,5129.1,2022,5642,23.4,Forte hausse,-9,-0.2
1,06088,4663,4576,4323,4416,4703,4763,4667,4803,4957,...,4651,4695,85661,4758.9,2021,5253,0.7,Stable,163,3.4
2,59350,4155,4224,4182,4081,4346,4499,4469,4466,4586,...,4334,4436,80527,4473.7,2020,4886,6.8,Hausse,225,4.8
3,33063,4155,4212,4178,4090,4306,4236,4379,4535,4221,...,4142,4108,76144,4230.2,2015,4535,-1.1,Stable,-124,-2.9
4,67482,3717,3743,3765,3753,3893,4188,4045,4213,4016,...,4380,4415,74441,4135.6,2020,4730,18.8,Hausse,619,15.1


In [22]:
url_deces = f"https://static.data.gouv.fr/resources/nombre-de-personnes-decedees-en-france-a-la-commune/20260330-074345/evolution-deces-par-commune-2008-2025.csv"
response = requests.get(url_deces)
if response.status_code == 200:
    with open("deces.csv", "wb") as file:
        file.write(response.content)
else :
    print('erreur')

deces2 = pd.read_csv('deces.csv', sep=';', dtype = str)
deces2.head()


,code_lieu_deces,deces_2008,deces_2009,deces_2010,deces_2011,deces_2012,deces_2013,deces_2014,deces_2015,deces_2016,...,deces_2024,deces_2025,total_2008_2025,moy_annuelle,annee_pic,deces_annee_pic,evol_pct_2008_vs_2025,tendance,surplus_covid_2020,surplus_pct_covid_2020
0,31555,4559,4647,4650,4763,4924,4877,4981,5208,5153,...,5433,5626,92323,5129.1,2022,5642,23.4,Forte hausse,-9,-0.2
1,06088,4663,4576,4323,4416,4703,4763,4667,4803,4957,...,4651,4695,85661,4758.9,2021,5253,0.7,Stable,163,3.4
2,59350,4155,4224,4182,4081,4346,4499,4469,4466,4586,...,4334,4436,80527,4473.7,2020,4886,6.8,Hausse,225,4.8
3,33063,4155,4212,4178,4090,4306,4236,4379,4535,4221,...,4142,4108,76144,4230.2,2015,4535,-1.1,Stable,-124,-2.9
4,67482,3717,3743,3765,3753,3893,4188,4045,4213,4016,...,4380,4415,74441,4135.6,2020,4730,18.8,Hausse,619,15.1


A présent, nous nous concentrons sur les données en Île de France. Pour cela nous allons créer une colonne département à partir de code_lieu_décès qui correspond au code de la commune. Vérifions tout d'abord la structure du dataframe. 

In [30]:
deces.info()

<class 'pandas.DataFrame'>
RangeIndex: 36765 entries, 0 to 36764
Data columns (total 27 columns):
 #   Column                  Non-Null Count  Dtype
---  ------                  --------------  -----
 0   code_lieu_deces         36765 non-null  str  
 1   deces_2008              36765 non-null  str  
 2   deces_2009              36765 non-null  str  
 3   deces_2010              36765 non-null  str  
 4   deces_2011              36765 non-null  str  
 5   deces_2012              36765 non-null  str  
 6   deces_2013              36765 non-null  str  
 7   deces_2014              36765 non-null  str  
 8   deces_2015              36765 non-null  str  
 9   deces_2016              36765 non-null  str  
 10  deces_2017              36765 non-null  str  
 11  deces_2018              36765 non-null  str  
 12  deces_2019              36765 non-null  str  
 13  deces_2020              36765 non-null  str  
 14  deces_2021              36765 non-null  str  
 15  deces_2022              36765 

On identifie les départements grâce aux deux premiers chiffres du code commune. Ensuite, on conserve uniquement les communes situées en Île de France

In [ ]:
deces = deces.rename({"code_lieu_deces": "code_commune"}, axis=1)

deces["dep"] = deces["code_commune"].str[:2]
deces["dep"]
#rq : 99 fait référence aux pays étrangers

0        31
1        06
2        59
3        33
4        67
         ..
36760    11
36761    77
36762    88
36763    80
36764    99
Name: dep, Length: 36765, dtype: str

In [39]:
dep_idf = ["75", "77", "78", "91", "92", "93", "94", "95"]
deces_idf = deces[deces["dep"].isin(dep_idf)].copy()

#on peut alors vérifier le nombre de communes obtenues par département : cela a l'air plutôt cohérent 
print(deces_idf["dep"].value_counts().sort_index())

dep
75     20
77    513
78    262
91    193
92     36
93     41
94     47
95    183
Name: count, dtype: int64


Afin de calculer un indice de taux de mortalité : il faut également avoir en tête que chaque commune ne compte pas le même nombre d'habitants. Ainsi, étudier l'évolution de la mortalité passe par étudier l'évolution de la démographie. 

Pour cela, nous téléchargons des données concernant les populations municipales (donc par communes). 

In [63]:
url_pop = f"https://www.data.gouv.fr/api/1/datasets/r/27361ae9-dcab-4312-97a1-48dab5f39218"
r = requests.get(url_pop)

In [ ]:
import io
import zipfile

z = zipfile.ZipFile(io.BytesIO(r.content))
print(z.namelist())   # les fichiers du zip

['DS_POPULATIONS_HISTORIQUES_metadata.csv', 'DS_POPULATIONS_HISTORIQUES_data.csv']


In [ ]:
#le fichier que nous souhaitons avoir est celui qui ne contient pas les metadata
fichier_data_pop = [n for n in z.namelist() if "metadata" not in n][0]

#on ouvre le zip
with z.open(fichier_data_pop) as f:
    pop = pd.read_csv(f, sep=";", dtype=str)

print(pop.shape)
print(pop.columns.tolist())
pop.head()

(813234, 6)
['FREQ', 'GEO', 'GEO_OBJECT', 'POPREF_MEASURE', 'TIME_PERIOD', 'OBS_VALUE']


,FREQ,GEO,GEO_OBJECT,POPREF_MEASURE,TIME_PERIOD,OBS_VALUE
0,A,15113,COM,PMUN,2010,197
1,A,14491,COM,PMUN,2007,154
2,A,14761,COM,PMUN,2011,746
3,A,14700,COM,PMUN,2018,663
4,A,15038,COM,PMUN,2013,1040


Nous allons conserver les geo_object étant des communes, soit COM, et ensuite se concentrer sur l'Île de France. 
Mais tout d'abord, il est possible d'observer dès les premières lignes que le tableau est dans un format particulier : en effet, une ligne correspond à une année observée. La même commune est donc présente plusieurs fois. 

Nous modifions le format du tableau afin d'obtenir une commune = une ligne. 

Pour cela, nous gardons GEO_OBJECT = COM, les communes, et une mesure par population municipale, POPREF_MEASURE = PMUN.

In [ ]:
pop_com = pop[(pop["POPREF_MEASURE"] == "PMUN") & (pop["GEO_OBJECT"] == "COM")].copy()

pop_com["GEO"] = pop_com["GEO"].astype(str).str.zfill(5)
pop_com["TIME_PERIOD"] = pop_com["TIME_PERIOD"].astype(int)
pop_com["OBS_VALUE"] = pd.to_numeric(pop_com["OBS_VALUE"], errors="coerce")

# on ne garde que la période utile par rapport aux données sur la mortalité 
pop_com = pop_com[pop_com["TIME_PERIOD"] >= 2008]

pop_wide = pop_com.pivot(index="GEO", columns="TIME_PERIOD", values="OBS_VALUE")
pop_wide.columns = [f"pop_{annee}" for annee in pop_wide.columns]
pop_wide = pop_wide.reset_index().rename(columns={"GEO": "code_commune"})

pop_wide.head()

,code_commune,pop_2008,pop_2009,pop_2010,pop_2011,pop_2012,pop_2013,pop_2014,pop_2015,pop_2016,pop_2017,pop_2018,pop_2019,pop_2020,pop_2021,pop_2022,pop_2023
0,01001,791,787,784,780,777,767,767,767,767,776,771,779,806,832,859,860
1,01002,194,207,221,234,235,236,239,241,243,248,253,256,262,267,273,270
2,01004,12792,13350,13835,13839,14233,14359,14022,14127,14081,14035,14204,14134,14288,14854,15554,15934
3,01005,1567,1592,1616,1600,1642,1635,1627,1619,1671,1689,1720,1751,1782,1897,1917,1906
4,01006,123,120,116,112,110,108,109,109,110,111,112,112,113,113,114,115


In [77]:
pop_wide.shape

(34858, 18)

In [75]:
pop_wide['dep'] = pop_wide['code_commune'].str[:2]
pop_idf = pop_wide[pop_wide["dep"].isin(dep_idf)].copy()
print(pop_idf["dep"].value_counts().sort_index())


dep
75      1
77    507
78    259
91    194
92     36
93     39
94     47
95    183
Name: count, dtype: int64


Il semble y avoir un problème avec paris : les arrondissements ont été absorbés. Ils correspondent en fait à un code particulier, ARM. Il faut adapter le code. 

In [81]:
pop_test = pop_wide[pop_wide['dep']=="75"]
pop_test

,code_commune,pop_2008,pop_2009,pop_2010,pop_2011,pop_2012,pop_2013,pop_2014,pop_2015,pop_2016,pop_2017,pop_2018,pop_2019,pop_2020,pop_2021,pop_2022,pop_2023,dep
29194,75056,2211297,2234105,2243833,2249975,2240621,2229621,2220445,2206488,2190327,2187526,2175601,2165423,2145906,2133111,2113705,2103778,75
